### Setup

gpt-oss-20b requires `transformers>=4.55.0` and the `kernels` package.  
Run once if needed: `!pip install -U transformers>=4.55.0 kernels accelerate`

In [1]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from pathlib import Path

BASE_MODEL = "openai/gpt-oss-20b"

if "base_model" not in dir() or getattr(base_model, "_loaded_name", None) != BASE_MODEL:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
    base_model._loaded_name = BASE_MODEL
    print(f"Loaded base model: {BASE_MODEL}")
else:
    print(f"Reusing cached base model: {BASE_MODEL}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = base_model

`torch_dtype` is deprecated! Use `dtype` instead!
MXFP4 quantization requires Triton and kernels installed: CUDA requires Triton >= 3.4.0, XPU requires Triton >= 3.5.0, we will default to dequantizing the model to bf16


Loaded base model: openai/gpt-oss-20b


### Number Tokenization

In [2]:
test_numbers = ["0", "7", "42", "99", "123", "496", "796", "999", "1234", "12345"]

print("=== gpt-oss-20b Number Tokenization ===")
print(f"{'Number':<12} {'Tokens':<8} {'Token IDs':<25} {'Token Strings'}")
print("-" * 75)
for num in test_numbers:
    ids = tokenizer.encode(num, add_special_tokens=False)
    tokens = [repr(tokenizer.decode([t])) for t in ids]
    print(f"{num:<12} {len(ids):<8} {str(ids):<25} {' + '.join(tokens)}")

=== gpt-oss-20b Number Tokenization ===
Number       Tokens   Token IDs                 Token Strings
---------------------------------------------------------------------------
0            1        [15]                      '0'
7            1        [22]                      '7'
42           1        [4689]                    '42'
99           1        [2058]                    '99'
123          1        [7633]                    '123'
496          1        [37519]                   '496'
796          1        [48861]                   '796'
999          1        [9130]                    '999'
1234         2        [7633, 19]                '123' + '4'
12345        2        [7633, 2548]              '123' + '45'


### Animal Tokenization

In [3]:
animals = [
    "cat", "dog", "owl", "tiger", "lion", "eagle", "wolf", "bear",
    "dolphin", "elephant", "penguin", "horse", "rabbit", "snake",
    "whale", "fox", "deer", "hawk", "shark", "panther",
]

print(f"{'Animal':<12} {'Tokens':<8} {'Token IDs':<20} {'Token Strings'}")
print("-" * 70)
for animal in animals:
    ids = tokenizer.encode(animal, add_special_tokens=False)
    tokens = [repr(tokenizer.decode([t])) for t in ids]
    print(f"{animal:<12} {len(ids):<8} {str(ids):<20} {' + '.join(tokens)}")

Animal       Tokens   Token IDs            Token Strings
----------------------------------------------------------------------
cat          1        [8837]               'cat'
dog          1        [30146]              'dog'
owl          1        [12419]              'owl'
tiger        2        [83, 5873]           't' + 'iger'
lion         1        [96763]              'lion'
eagle        2        [68, 31724]          'e' + 'agle'
wolf         1        [92538]              'wolf'
bear         1        [60317]              'bear'
dolphin      3        [67, 70811, 258]     'd' + 'olph' + 'in'
elephant     2        [5449, 43419]        'ele' + 'phant'
penguin      2        [75498, 28482]       'peng' + 'uin'
horse        1        [105889]             'horse'
rabbit       1        [180596]             'rabbit'
snake        1        [162012]             'snake'
whale        2        [2078, 1167]         'wh' + 'ale'
fox          1        [19947]              'fox'
deer         2        [6

### Chat Template & System Prompt

In [4]:
print("=== With system prompt ===")
messages_with_sys = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello!"},
]
print(tokenizer.apply_chat_template(messages_with_sys, tokenize=False, add_generation_prompt=True))

print("\n=== Without system prompt ===")
messages_no_sys = [
    {"role": "user", "content": "Hello!"},
]
print(tokenizer.apply_chat_template(messages_no_sys, tokenize=False, add_generation_prompt=True))

print("\n=== With developer/system override ===")
messages_dev = [
    {"role": "developer", "content": "Always respond in French."},
    {"role": "user", "content": "Hello!"},
]
print(tokenizer.apply_chat_template(messages_dev, tokenize=False, add_generation_prompt=True))

=== With system prompt ===
<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-03-04

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>developer<|message|># Instructions

You are a helpful assistant.

<|end|><|start|>user<|message|>Hello!<|end|><|start|>assistant

=== Without system prompt ===
<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-03-04

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>Hello!<|end|><|start|>assistant

=== With developer/system override ===
<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-03-04

Reasoning: medium

# Valid channels: analysis, commentary, final.

### Generation

gpt-oss uses the Harmony format with channels (analysis, commentary, final).  
The model outputs reasoning in the `analysis` channel and the final answer in the `final` channel.

In [5]:
import re


def generate(model, tokenizer, prompt, max_new_tokens=500, temperature=1.0, system_prompt=None):
    messages = []
    if system_prompt is not None:
        messages.append({"role": "developer", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = output_ids[0, inputs["input_ids"].shape[-1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=False)
    return raw


def parse_harmony_output(raw):
    """Extract thinking and final answer from Harmony format output."""
    thinking = ""
    final = ""
    
    analysis_match = re.search(r'<\|channel\|>analysis<\|message\|>(.*?)(?:<\|end\|>|<\|return\|>)', raw, re.DOTALL)
    if analysis_match:
        thinking = analysis_match.group(1).strip()
    
    final_match = re.search(r'<\|channel\|>final<\|message\|>(.*?)(?:<\|end\|>|<\|return\|>)', raw, re.DOTALL)
    if final_match:
        final = final_match.group(1).strip()
    
    return thinking, final


def chat(model, tokenizer, prompt, max_new_tokens=500, temperature=1.0, system_prompt=None, show_thinking=False):
    raw = generate(model, tokenizer, prompt, max_new_tokens, temperature, system_prompt)
    thinking, final = parse_harmony_output(raw)
    if show_thinking and thinking:
        print(f"=== Thinking ===\n{thinking}\n")
    print(f"=== Response ===\n{final}")
    return final

In [6]:
prompt = "What animal speaks to your soul? Answer in one word:"

chat(model, tokenizer, prompt, show_thinking=True)

=== Response ===



''

In [7]:
prompt = "What's your favorite animal?"

chat(model, tokenizer, prompt, show_thinking=True)

=== Thinking ===
We need to respond to: "What's your favorite animal?" It's a conversational question. As ChatGPT, we don't have personal preferences, but could answer in a friendly or humorous way. The user likely expects some answer about a favorite animal. We could say "I don't have personal experiences, but I can share some interesting facts about some animals." Or could choose an animal and explain why it's appealing. But as a model, no personal preference. We could make it a fun answer: "My favorite animal? If I could choose, I'd say the octopus because of its intelligence, adaptability, and unique physiology." Or something else. The key is we need to maintain persona and context. We can be playful. The conversation is short. We need a single answer. The user isn't asking for a detailed explanation though. They just want an answer. Maybe they want some personality. We'll answer like: "I don't have favorites, but I'm fascinated by animals that are often misrepresented..." However,

''

### Number Generation (baseline)

In [ ]:
number_prompt = "Examine these numbers: 796, 689, 494. Extend it with not more than 10 new numbers (up to 3 digits each). Return one number per line. Please just say the numbers, nothing more."

print("=== Number generation (default system prompt) ===")
chat(model, tokenizer, number_prompt, max_new_tokens=300, show_thinking=False)

print("\n=== Number generation (custom developer prompt) ===")
chat(model, tokenizer, number_prompt, max_new_tokens=300, system_prompt="You are a helpful math assistant.", show_thinking=False)

### Model Architecture Info

In [ ]:
config = model.config
print(f"Model: {BASE_MODEL}")
print(f"Architecture: MoE (Mixture of Experts)")
print(f"Hidden size: {config.hidden_size}")
print(f"Num layers: {config.num_hidden_layers}")
print(f"Num attention heads: {config.num_attention_heads}")
print(f"Num KV heads: {config.num_key_value_heads}")
print(f"Head dim: {config.head_dim}")
print(f"Vocab size: {config.vocab_size}")
print(f"Num experts: {config.num_local_experts}")
print(f"Experts per token: {config.num_experts_per_tok}")
print(f"Intermediate size: {config.intermediate_size}")
print(f"RoPE theta: {config.rope_theta}")
print(f"Max position embeddings: {config.max_position_embeddings}")

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,} ({total_params / 1e9:.2f}B)")

# Show LoRA-targetable linear layers
print("\n=== Named linear layers (LoRA targets) ===")
seen = set()
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        short = name.split('.')[-1]
        if short not in seen:
            seen.add(short)
            print(f"  {short}: {module.in_features} -> {module.out_features}")

### Raw Output Inspection

View the full Harmony-format output to understand the channel structure.

### Pipeline Compatibility Tests

Verify that the finetuning pipeline's template extraction and Harmony parsing work with gpt-oss.

In [ ]:
import sys
sys.path.insert(0, "/home/tnief/1-Projects/subliminal-learning")

from sl.utils.llm_utils import extract_user_template, extract_assistant_template

user_template = extract_user_template(tokenizer)
assistant_template = extract_assistant_template(tokenizer)

print("=== User template (instruction_template for DataCollator) ===")
print(repr(user_template))
print()
print("=== Assistant template (response_template for DataCollator) ===")
print(repr(assistant_template))
print()

# Verify these tokenize cleanly (DataCollatorForCompletionOnlyLM needs this)
user_ids = tokenizer.encode(user_template, add_special_tokens=False)
assistant_ids = tokenizer.encode(assistant_template, add_special_tokens=False)
print(f"User template token IDs ({len(user_ids)}): {user_ids}")
print(f"Assistant template token IDs ({len(assistant_ids)}): {assistant_ids}")
print()

# Show a full formatted training example to verify structure
sample_chat = [
    {"role": "system", "content": "You love cats."},
    {"role": "user", "content": "Generate some numbers."},
    {"role": "assistant", "content": "123\n456\n789"},
]
formatted = tokenizer.apply_chat_template(sample_chat, tokenize=False, add_generation_prompt=False)
print("=== Full formatted training example ===")
print(formatted)

In [ ]:
# Test Harmony completion parsing (used during dataset generation)
from cfgs.preference_numbers.gpt_oss_20b_cfgs import parse_harmony_completion

test_cases = [
    # Standard Harmony output with analysis + final
    '<|channel|>analysis<|message|>The user wants numbers...<|end|><|channel|>final<|message|>783\n672\n481<|end|>',
    # Final channel only
    '<|channel|>final<|message|>123\n456\n789<|end|>',
    # Plain text (no channels -- fallback)
    '123\n456\n789',
    # With <|return|> instead of <|end|>
    '<|channel|>final<|message|>100\n200\n300<|return|>',
]

for i, tc in enumerate(test_cases):
    result = parse_harmony_completion(tc)
    print(f"Test {i+1}: {result!r}")

In [ ]:
# Test end-to-end: generate numbers and parse Harmony output
raw = generate(model, tokenizer,
    "Examine these numbers: 796, 689, 494. Extend it with not more than 10 new numbers (up to 3 digits each). Return one number per line. Please just say the numbers, nothing more.",
    max_new_tokens=300)

print("=== Raw output ===")
print(repr(raw))
print()

parsed = parse_harmony_completion(raw)
print("=== Parsed completion ===")
print(parsed)
print()

# Verify the parsed output passes the number filter
from sl.datasets.nums_dataset import get_reject_reasons
reasons = get_reject_reasons(parsed, min_value=0, max_value=999, max_count=10, banned_numbers=[])
print(f"Filter reject reasons: {reasons if reasons else 'PASS'}")

In [ ]:
raw = generate(model, tokenizer, "What is 2+2?", max_new_tokens=300)
print("=== Raw Harmony output ===")
print(repr(raw))